In [4]:
import collections, hashlib, json, math, re, urllib.request
from pathlib import Path
URL='https://raw.githubusercontent.com/yancyanq/fuzzy-nli-adaptive-retrieval/main/exp/step1/pilot.json'
STOP=set('a an the of in on at to for and or is are was were be been by with from as that this which who whom what when where how did does do it its his her their they he she'.split())
def tok(s):
    return [w for w in re.findall(r'[a-z0-9]+',s.lower()) if w not in STOP]
def score(corpus,query):
    docs=[collections.Counter(tok(s)) for s in corpus]
    n=len(docs); lengths=[sum(d.values()) for d in docs]; avg=sum(lengths)/n
    df=collections.Counter(w for d in docs for w in d)
    idf={w:math.log(n-f+0.5)-math.log(f+0.5) for w,f in df.items()}
    floor=0.25*sum(idf.values())/len(idf) if idf else 0
    idf={w:(floor if v<0 else v) for w,v in idf.items()}
    return [sum(idf.get(w,0)*((d[w]*2.5)/(d[w]+1.5*(0.25+0.75*l/avg))) for w in tok(query)) for d,l in zip(docs,lengths)]
def ordering(corpus,query):
    scores=score(corpus,query)
    return sorted(range(len(corpus)),key=lambda i:-scores[i])
raw=urllib.request.urlopen(URL,timeout=90).read()
data=json.loads(raw)[:50]
assert len(data)==50 and len({e['_id'] for e in data})==50
rows=[]
for ex in data:
    ctx=ex['context']; titles=[x[0] for x in ctx]
    corpus=[t+' '+' '.join(ss) for t,ss in ctx]
    gold=set(t for t,_ in ex['supporting_facts']) & set(titles)
    assert gold
    order=ordering(corpus,ex['question']); first=order[:2]
    top=ctx[order[0]]; query=ex['question']+' '+top[0]+' '+(top[1][0] if top[1] else '')
    unused=[i for i in range(len(corpus)) if i not in first]
    more=[unused[j] for j in ordering([corpus[i] for i in unused],query)]
    row={'id':ex['_id']}
    for k in (2,4,6): row['top'+str(k)]=int(gold <= {titles[i] for i in order[:k]})
    for m in (1,2,4): row['2plus'+str(m)]=int(gold <= {titles[i] for i in first+more[:m]})
    rows.append(row)
result={'scope':'exploratory, first 50 prior pilot questions, 10 paragraph candidate pool; reference paragraph title coverage only; no LLM inference','n':len(rows),'data_sha256':hashlib.sha256(raw).hexdigest(),'coverage':{key:sum(r[key] for r in rows)/len(rows) for key in rows[0] if key!='id'},'rows':rows}
print(json.dumps({k:v for k,v in result.items() if k!='rows'},indent=2))
Path('/kaggle/working/v3_retrieval_preflight.json').write_text(json.dumps(result,indent=2))


{
  "scope": "exploratory, first 50 prior pilot questions, 10 paragraph candidate pool; reference paragraph title coverage only; no LLM inference",
  "n": 50,
  "data_sha256": "673a370110adf24bd54956467010c5723a6facb7ef708997d088b9c2c627f790",
  "coverage": {
    "top2": 0.32,
    "top4": 0.58,
    "top6": 0.7,
    "2plus1": 0.66,
    "2plus2": 0.78,
    "2plus4": 0.92
  }
}


8393

In [5]:
import random, time
start=time.monotonic()
TUNE_URL='https://raw.githubusercontent.com/yancyanq/fuzzy-nli-adaptive-retrieval/main/exp/step1/tune.json'
tune_raw=urllib.request.urlopen(TUNE_URL,timeout=120).read()
tune=json.loads(tune_raw)
# Build a shared donor pool from held-in development material. Never use final-test items.
donors=[]; seen=set()
for ex in tune:
    for title, sentences in ex['context']:
        key=(title.casefold().strip(), ' '.join(sentences).strip())
        if key not in seen:
            seen.add(key); donors.append((title,sentences))
summary={str(n):[] for n in (10,30,60)}
for ex in data:
    original=ex['context']; original_titles={t.casefold().strip() for t,_ in original}
    eligible=[x for x in donors if x[0].casefold().strip() not in original_titles]
    seed=int(hashlib.sha256(('v3-preflight-42:'+ex['_id']).encode()).hexdigest()[:16],16)
    rng=random.Random(seed)
    subset=rng.sample(eligible,min(500,len(eligible)))
    texts=[t+' '+' '.join(ss) for t,ss in subset]
    ranked=ordering(texts,ex['question'])
    hard=[subset[i] for i in ranked[:25]]
    remaining=[subset[i] for i in ranked[25:]]
    random_extra=rng.sample(remaining,25)
    extras=[item for pair in zip(hard,random_extra) for item in pair]
    for n in (10,30,60):
        ctx=list(original)+extras[:n-10]
        titles=[t for t,_ in ctx]; corpus=[t+' '+' '.join(ss) for t,ss in ctx]
        rank=ordering(corpus,ex['question']); first=rank[:2]
        top=ctx[rank[0]]; query=ex['question']+' '+top[0]+' '+(top[1][0] if top[1] else '')
        unused=[i for i in range(len(ctx)) if i not in first]
        more=[unused[j] for j in ordering([corpus[i] for i in unused],query)]
        gold=set(t for t,_ in ex['supporting_facts']) & set(titles)
        assert gold
        row={'id':ex['_id']}
        for k in (2,4,6): row['top'+str(k)]=int(gold <= {titles[i] for i in rank[:k]})
        for m in (1,2,4): row['2plus'+str(m)]=int(gold <= {titles[i] for i in first+more[:m]})
        summary[str(n)].append(row)
result2={'scope':'exploratory: same 50 previously used pilot questions; augmented candidate pools from 500 randomly sampled donor paragraphs per question; half BM25-ranked hard and half random; no LLM inference; gold title coverage only','pilot_sha256':hashlib.sha256(raw).hexdigest(),'donor_sha256':hashlib.sha256(tune_raw).hexdigest(),'donor_unique':len(donors),'seconds':round(time.monotonic()-start,2),'coverage':{n:{k:sum(r[k] for r in rs)/len(rs) for k in rs[0] if k!='id'} for n,rs in summary.items()},'rows':summary}
print(json.dumps({k:v for k,v in result2.items() if k!='rows'},indent=2))
Path('/kaggle/working/v3_expanded_preflight.json').write_text(json.dumps(result2,indent=2))


{
  "scope": "exploratory: same 50 previously used pilot questions; augmented candidate pools from 500 randomly sampled donor paragraphs per question; half BM25-ranked hard and half random; no LLM inference; gold title coverage only",
  "pilot_sha256": "673a370110adf24bd54956467010c5723a6facb7ef708997d088b9c2c627f790",
  "donor_sha256": "478f6b2f4d7506a0b8d804de5aa077eacdd3ef20d90f127ac8d22999453ea53b",
  "donor_unique": 4948,
  "seconds": 2.78,
  "coverage": {
    "10": {
      "top2": 0.32,
      "top4": 0.58,
      "top6": 0.7,
      "2plus1": 0.66,
      "2plus2": 0.78,
      "2plus4": 0.92
    },
    "30": {
      "top2": 0.36,
      "top4": 0.54,
      "top6": 0.62,
      "2plus1": 0.62,
      "2plus2": 0.76,
      "2plus4": 0.88
    },
    "60": {
      "top2": 0.34,
      "top4": 0.5,
      "top6": 0.56,
      "2plus1": 0.6,
      "2plus2": 0.68,
      "2plus4": 0.86
    }
  }
}


27669

In [3]:
for n, values in result2['coverage'].items():
    print(n, ' '.join(k+'='+str(round(100*v))+'%' for k,v in values.items()))


10 top2=32% top4=58% top6=70% 2plus1=66% 2plus2=78% 2plus4=92%
30 top2=36% top4=54% top6=62% 2plus1=62% 2plus2=76% 2plus4=88%
60 top2=34% top4=50% top6=56% 2plus1=60% 2plus2=68% 2plus4=86%


In [1]:
import sys, importlib, torch
print('Python:',sys.version.split()[0], 'Torch:',torch.__version__, 'CUDA:',torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    p=torch.cuda.get_device_properties(i)
    print('GPU',i,p.name,round(p.total_memory/1024**3,2),'GiB')
for name in ['transformers','accelerate','safetensors']:
    try:
        m=importlib.import_module(name); print(name,getattr(m,'__version__','available'))
    except Exception as e: print(name,'UNAVAILABLE',repr(e))


Python: 3.13.16 Torch: 2.11.0+cu130 CUDA: True
GPU 0 Tesla T4 14.56 GiB
GPU 1 Tesla T4 14.56 GiB
transformers 5.20.2
accelerate 1.15.0
safetensors 0.8.0


In [2]:
import time, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
MODEL='Qwen/Qwen2.5-7B-Instruct'
REV='a09a35458c702b33eeacc393d103063234e8bc28'
t0=time.monotonic()
print('Loading tokenizer and official model revision...', flush=True)
tokenizer=AutoTokenizer.from_pretrained(MODEL, revision=REV)
model=AutoModelForCausalLM.from_pretrained(MODEL, revision=REV, dtype=torch.float16, device_map='auto', max_memory={0:'13GiB',1:'13GiB','cpu':'20GiB'})
model.eval()
print('Model loaded in seconds:',round(time.monotonic()-t0,1), 'device_map:',model.hf_device_map,flush=True)
messages=[{'role':'user','content':'Based only on the evidence, answer briefly. Evidence: The director of Star Sea is Lin Zhou. Lin Zhou was born in Haicheng. Question: Which city was the director born in?'}]
batch=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt',return_dict=True).to('cuda:0')
with torch.inference_mode():
    generated=model.generate(**batch,max_new_tokens=32,do_sample=False)
print('Generated:',tokenizer.decode(generated[0][batch['input_ids'].shape[-1]:],skip_special_tokens=True))
print('Input tokens:',batch['input_ids'].shape[-1], 'Output tokens:',generated.shape[-1]-batch['input_ids'].shape[-1])


Loading tokenizer and official model revision...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (3 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Model loaded in seconds: 90.8 device_map: {'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 0, 'model.layers.5': 0, 'model.layers.6': 0, 'model.layers.7': 0, 'model.layers.8': 0, 'model.layers.9': 0, 'model.layers.10': 0, 'model.layers.11': 0, 'model.layers.12': 0, 'model.layers.13': 0, 'model.layers.14': 1, 'model.layers.15': 1, 'model.layers.16': 1, 'model.layers.17': 1, 'model.layers.18': 1, 'model.layers.19': 1, 'model.layers.20': 1, 'model.layers.21': 1, 'model.layers.22': 1, 'model.layers.23': 1, 'model.layers.24': 1, 'model.layers.25': 1, 'model.layers.26': 1, 'model.layers.27': 1, 'model.norm': 1, 'model.rotary_emb': 1, 'lm_head': 1}


AttributeError: 

In [3]:
batch=tokenizer.apply_chat_template(messages,tokenize=True,add_generation_prompt=True,return_tensors='pt',return_dict=True).to('cuda:0')
with torch.inference_mode():
    generated=model.generate(**batch,max_new_tokens=32,do_sample=False)
print('Generated:',tokenizer.decode(generated[0][batch['input_ids'].shape[-1]:],skip_special_tokens=True))
print('Input tokens:',batch['input_ids'].shape[-1], 'Output tokens:',generated.shape[-1]-batch['input_ids'].shape[-1])


Generated: Haicheng.
Input tokens: 68 Output tokens: 5


In [6]:
import string, collections, time, random

def normalise(s):
    s=s.lower(); s=''.join(ch for ch in s if ch not in set(string.punctuation))
    s=re.sub(r'\b(a|an|the)\b',' ',s)
    return ' '.join(s.split())
def answer_f1(pred,truth):
    p=normalise(pred); t=normalise(truth)
    if (p in {'yes','no','noanswer'} or t in {'yes','no','noanswer'}) and p!=t: return 0.0
    a=p.split(); b=t.split(); common=sum((collections.Counter(a)&collections.Counter(b)).values())
    if not common: return float(not a and not b)
    return 2*common/(len(a)+len(b))
def make_pool(ex,n=30):
    original=ex['context']; titles={t.casefold().strip() for t,_ in original}
    eligible=[x for x in donors if x[0].casefold().strip() not in titles]
    seed=int(hashlib.sha256(('v3-preflight-42:'+ex['_id']).encode()).hexdigest()[:16],16)
    rng=random.Random(seed); subset=rng.sample(eligible,min(500,len(eligible)))
    ranked=ordering([t+' '+' '.join(ss) for t,ss in subset],ex['question'])
    hard=[subset[i] for i in ranked[:25]]; remaining=[subset[i] for i in ranked[25:]]
    extras=[x for pair in zip(hard,rng.sample(remaining,25)) for x in pair]
    return list(original)+extras[:n-10]
def contexts_for(ex):
    ctx=make_pool(ex); corpus=[t+' '+' '.join(ss) for t,ss in ctx]
    rank=ordering(corpus,ex['question']); first=rank[:2]; top=ctx[rank[0]]
    q=ex['question']+' '+top[0]+' '+(top[1][0] if top[1] else '')
    unused=[i for i in range(len(ctx)) if i not in first]
    more=[unused[j] for j in ordering([corpus[i] for i in unused],q)]
    return ctx,{'top2':first,'top6':rank[:6],'2plus2':first+more[:2],'2plus4':first+more[:4],'full30':list(range(len(ctx)))}
records=[]
for ex in data[:10]:
    ctx, strategies=contexts_for(ex)
    for strategy, idx in strategies.items():
        passages='\n'.join('['+str(i)+'] '+ctx[i][0]+': '+' '.join(ctx[i][1]) for i in idx)
        prompt='Answer the question using only the provided evidence. Return a short answer only. If unsupported, say unknown.\nEvidence:\n'+passages+'\nQuestion: '+ex['question']+'\nAnswer:'
        enc=tokenizer.apply_chat_template([{'role':'user','content':prompt}],tokenize=True,add_generation_prompt=True,return_tensors='pt',return_dict=True).to('cuda:0')
        n=enc['input_ids'].shape[-1]; rec={'id':ex['_id'],'strategy':strategy,'input_tokens':n,'selected_paragraphs':len(idx)}
        if n+32>4096:
            rec.update(status='context_exceeds_4096')
        else:
            try:
                torch.cuda.synchronize(); start=time.monotonic()
                with torch.inference_mode(): out=model.generate(**enc,max_new_tokens=32,do_sample=False)
                torch.cuda.synchronize(); seconds=time.monotonic()-start
                answer=tokenizer.decode(out[0][n:],skip_special_tokens=True).strip()
                rec.update(status='ok',answer=answer,reference=ex['answer'],f1=answer_f1(answer,ex['answer']),output_tokens=out.shape[-1]-n,seconds=seconds)
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); rec.update(status='gpu_oom')
        records.append(rec)
        Path('/kaggle/working/v3_qwen_10q_pilot.json').write_text(json.dumps(records,indent=2))
        print(len(records),strategy,rec['status'], 'input',n, 'f1',rec.get('f1','-'),flush=True)
print('EXPLORATORY ONLY: first 10 reused pilot questions, 30 candidate passages, direct answers, no NLI or fuzzy gate; full30 excluded if over 4096 tokens.')


1 top2 ok input 361 f1 1.0
2 top6 ok input 662 f1 1.0
3 2plus2 ok input 506 f1 1.0
4 2plus4 ok input 799 f1 1.0
5 full30 context_exceeds_4096 input 4312 f1 -
6 top2 ok input 251 f1 0.0
7 top6 ok input 699 f1 0.0
8 2plus2 ok input 476 f1 0.0
9 2plus4 ok input 699 f1 0.0
10 full30 context_exceeds_4096 input 4122 f1 -
11 top2 ok input 286 f1 0.0
12 top6 ok input 1115 f1 0.0
13 2plus2 ok input 617 f1 0.0
14 2plus4 ok input 1006 f1 0.8
15 full30 context_exceeds_4096 input 4979 f1 -
16 top2 ok input 354 f1 1.0
17 top6 ok input 967 f1 0.36363636363636365
18 2plus2 ok input 699 f1 0.6666666666666666
19 2plus4 ok input 1204 f1 0.26666666666666666
20 full30 context_exceeds_4096 input 4723 f1 -
21 top2 ok input 339 f1 0.0
22 top6 ok input 897 f1 0.0
23 2plus2 ok input 677 f1 0.8
24 2plus4 ok input 914 f1 0.0
25 full30 context_exceeds_4096 input 4565 f1 -
26 top2 ok input 345 f1 0.0
27 top6 ok input 911 f1 0.0
28 2plus2 ok input 565 f1 0.0
29 2plus4 ok input 819 f1 0.0
30 full30 ok input 3955 f1 0

In [7]:
for method in ['top2','top6','2plus2','2plus4','full30']:
    allrows=[r for r in records if r['strategy']==method]
    ok=[r for r in allrows if r['status']=='ok']
    print(method,'ok',len(ok),'/',len(allrows),'avg_f1',round(sum(r['f1'] for r in ok)/len(ok),3) if ok else None,'avg_input_tokens',round(sum(r['input_tokens'] for r in ok)/len(ok)) if ok else None,'skips',collections.Counter(r['status'] for r in allrows if r['status']!='ok'))


top2 ok 10 / 10 avg_f1 0.457 avg_input_tokens 318 skips Counter()
top6 ok 10 / 10 avg_f1 0.365 avg_input_tokens 880 skips Counter()
2plus2 ok 10 / 10 avg_f1 0.604 avg_input_tokens 578 skips Counter()
2plus4 ok 10 / 10 avg_f1 0.564 avg_input_tokens 886 skips Counter()
full30 ok 2 / 10 avg_f1 0.5 avg_input_tokens 3966 skips Counter({'context_exceeds_4096': 8})


In [8]:
from transformers import AutoModelForSequenceClassification
torch.set_num_threads(4)
NLI='MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli'
NLI_REV='b3546ea6b0346eb6f8d5d68b13c7dc6d0376b3d7'
start=time.monotonic()
print('Loading pinned NLI on CPU...',flush=True)
nli_tok=AutoTokenizer.from_pretrained(NLI,revision=NLI_REV)
nli_model=AutoModelForSequenceClassification.from_pretrained(NLI,revision=NLI_REV,dtype=torch.float32).to('cpu').eval()
print('NLI loaded seconds',round(time.monotonic()-start,2),'labels',nli_model.config.id2label,flush=True)
cases=[('Lin Zhou gave the letter to Lin Yue.','Lin Zhou gave the letter to Lin Yue.'),('Lin Zhou gave the letter to Lin Yue.','Lin Yue settled in Haicheng.'),('Lin Yue settled in Jiangcheng.','Lin Yue settled in Haicheng.')]
check=[]
for premise,claim in cases:
    encoded=nli_tok(premise,claim,return_tensors='pt',truncation=True,max_length=512)
    with torch.inference_mode(): probs=torch.softmax(nli_model(**encoded).logits,dim=-1)[0].tolist()
    row={'premise':premise,'claim':claim,'scores':{str(nli_model.config.id2label[i]):round(float(v),4) for i,v in enumerate(probs)}}
    check.append(row); print(row,flush=True)
Path('/kaggle/working/v3_nli_cpu_smoke.json').write_text(json.dumps(check,indent=2))


Loading pinned NLI on CPU...


config.json:   0%|          | 0.00/1.06k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.65M [00:00<?, ?B/s]

... (3 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

NLI loaded seconds 10.25 labels {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
{'premise': 'Lin Zhou gave the letter to Lin Yue.', 'claim': 'Lin Zhou gave the letter to Lin Yue.', 'scores': {'entailment': 0.9985, 'neutral': 0.0013, 'contradiction': 0.0002}}
{'premise': 'Lin Zhou gave the letter to Lin Yue.', 'claim': 'Lin Yue settled in Haicheng.', 'scores': {'entailment': 0.0006, 'neutral': 0.9987, 'contradiction': 0.0006}}
{'premise': 'Lin Yue settled in Jiangcheng.', 'claim': 'Lin Yue settled in Haicheng.', 'scores': {'entailment': 0.0103, 'neutral': 0.0083, 'contradiction': 0.9815}}


646